# Indian Herbal Leaf Classification Using a Custom CNN
### College Project | Deep Learning Computer Vision System

Based on the research paper: *"Leaf Classification Using A Custom CNN"* (Asha M Tarsadia Institute of Computer Science and Technology).

**Objective**: Automatically identify 5 Indian herbal plant leaf species from images:
1. **Lemon** (*Citrus limon*)
2. **Mango** (*Mangifera indica*)
3. **Rosa-senensis / Hibiscus** (*Hibiscus rosa-sinensis*)
4. **Tulsi** (*Ocimum tenuiflorum*)
5. **Money Plant** (*Epipremnum aureum*)

---
### Hardware Setup (Free T4 GPU):
Before running, enable GPU acceleration:
- Navigate to **Runtime** > **Change runtime type**
- Set **Hardware accelerator** to **T4 GPU**
- Click **Save**

## 1. Verify GPU & Environment Setup

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

## 2. Dataset Setup
Upload `Leaf_Dataset.zip` (included in your project folder) or mount Google Drive.

In [ ]:
import os
from google.colab import files

CLASS_NAMES = ['Lemon', 'Mango', 'Rosa-Senensis', 'Tulsi', 'money']
DATA_DIR = None

# Check for existing zip files
for candidate_zip in ['Leaf_Dataset.zip', 'Leaf.zip', 'leaf_dataset.zip']:
    if os.path.exists(candidate_zip):
        !unzip -qo {candidate_zip} -d /content/Leaf_Dataset
        DATA_DIR = '/content/Leaf_Dataset'
        break

# Check for Google Drive mount
if not DATA_DIR and os.path.isdir('/content/drive/MyDrive/Leaf'):
    DATA_DIR = '/content/drive/MyDrive/Leaf'

# If still not found, prompt for upload
if not DATA_DIR:
    print("\n--- Leaf_Dataset.zip not detected on Colab disk ---")
    print("Please select and upload 'Leaf_Dataset.zip' from your project directory:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.lower().endswith('.zip'):
            !unzip -qo "{fn}" -d /content/Leaf_Dataset
            DATA_DIR = '/content/Leaf_Dataset'
            break

if not DATA_DIR:
    raise FileNotFoundError("Dataset not found! Please upload Leaf_Dataset.zip to train the model.")

print(f"\nDataset ready at: {DATA_DIR}")
print("Class folders found:", [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

## 3. Data Pipeline & Class Weighting
We calculate smoothed balanced weights to ensure minority classes are fairly represented, and apply rotation, zoom, and horizontal flip augmentations.

In [ ]:
import numpy as np
from tensorflow.keras import layers, models

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
VAL_SPLIT = 0.2
SEED = 42

# Compute class counts and smoothed weights
class_counts = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATA_DIR, cls)
    cnt = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) if os.path.isdir(cls_path) else 1
    class_counts.append(max(cnt, 1))

total_samples = sum(class_counts)
raw_w = [total_samples / (len(CLASS_NAMES) * c) for c in class_counts]
sqrt_w = [np.sqrt(w) for w in raw_w]
norm_factor = len(CLASS_NAMES) / sum(sqrt_w)
class_weights = {i: float(sqrt_w[i] * norm_factor) for i in range(len(CLASS_NAMES))}

print("=" * 60)
print("Dataset Class Distribution & Smoothed Weights:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  • {cls:<15}: {class_counts[i]:4d} images | Weight: {class_weights[i]:.2f}")
print("=" * 60)

# Load train and validation sets
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='categorical',
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset='training'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='categorical',
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset='validation'
)

# Data augmentation followed safely by rescaling
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.Rescaling(1./255)
])

val_rescale = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (val_rescale(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

## 4. Custom CNN Model Architecture
Implementing the exact 3-block convolutional neural network architecture described in the research paper.

In [ ]:
def build_model():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        # Conv Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='valid'),
        layers.MaxPooling2D((2, 2)),
        # Conv Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='valid'),
        layers.MaxPooling2D((2, 2)),
        # Conv Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='valid'),
        layers.MaxPooling2D((2, 2)),
        # Classification Head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(5, activation='softmax')
    ], name="Custom_Leaf_CNN")
    return model

model = build_model()
model.summary()

## 5. Model Compilation & Training
Train using Adam optimizer and Categorical Cross-Entropy with checkpointing on validation accuracy.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint('leaf_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

EPOCHS = 15
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

## 6. Training Curves & Performance Visualization

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='#2563EB', lw=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='#16A34A', lw=2, ls='--')
plt.title('Accuracy over Epochs', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='#DC2626', lw=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='#D97706', lw=2, ls='--')
plt.title('Loss over Epochs', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 7. Model Evaluation & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Greens')
plt.title('Confusion Matrix - Herbal Leaf Classification', fontweight='bold')
plt.xticks(range(5), CLASS_NAMES, rotation=30)
plt.yticks(range(5), CLASS_NAMES)
for i in range(5):
    for j in range(5):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black', fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.tight_layout()
plt.show()

## 8. Interactive Leaf Prediction (Upload an Image)
Upload any leaf image to classify it and display its botanical and medicinal properties.

In [ ]:
from google.colab import files
from PIL import Image
import io

CLASS_INFO = {
    'Lemon': {'sci': 'Citrus limon', 'uses': 'Rich in Vitamin C, digestive aid, antimicrobial.'},
    'Mango': {'sci': 'Mangifera indica', 'uses': 'Rich in mangiferin, used in traditional remedies for blood sugar.'},
    'Rosa-Senensis': {'sci': 'Hibiscus rosa-sinensis', 'uses': 'Ayurvedic tonic for hair health, skin, and blood pressure.'},
    'Tulsi': {'sci': 'Ocimum tenuiflorum', 'uses': 'Sacred adaptogen herb, boosts immunity, treats cough & respiratory illness.'},
    'money': {'sci': 'Epipremnum aureum', 'uses': 'NASA-approved air purifying plant; filters VOC toxins.'}
}

print("Please upload a leaf image (.jpg / .png):")
uploaded = files.upload()
for fn in uploaded.keys():
    img = Image.open(io.BytesIO(uploaded[fn])).convert('RGB')
    img_resized = img.resize((128, 128))
    tensor = np.expand_dims(np.array(img_resized, dtype=np.float32) / 255.0, axis=0)
    
    preds = model.predict(tensor, verbose=0)[0]
    top_idx = int(np.argmax(preds))
    pred_cls = CLASS_NAMES[top_idx]
    conf = preds[top_idx] * 100
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Uploaded: {fn}\nPrediction: {pred_cls} ({conf:.1f}%)", fontweight='bold')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.barh(CLASS_NAMES, preds * 100, color='#16A34A')
    plt.xlim(0, 100)
    plt.xlabel('Confidence (%)')
    plt.title('Class Probabilities')
    plt.tight_layout()
    plt.show()
    
    print(f"Species: {pred_cls} ({CLASS_INFO[pred_cls]['sci']})")
    print(f"Medicinal Properties: {CLASS_INFO[pred_cls]['uses']}")